# SB3-based Race Track Training Notebook

This notebook replaces the old TensorFlow training loop with Stable-Baselines3 (SB3) and reuses the custom HighwayEnv race-track environment.

What the agent sees:
- Observations: the environment exposes an occupancy-grid style observation around the ego vehicle. In this wrapper it is a 2x18x18 tensor-like array: the first channel marks road presence and the second marks whether the cell is on-road.
- Actions: PPO uses a continuous action vector [steering, throttle]. In this environment the throttle is effectively kept near zero, so the policy mainly learns steering. DQN uses a discrete steering action with three options: left, straight, right.
- Reward: the environment rewards progress around the track, staying near the lane center, and avoiding large steering costs. It penalizes going off-road or crashing, so better episodes achieve higher cumulative reward.

The code follows the same spirit as the HighwayEnv DQN example and the racetrack PPO example: keep the training loop simple, log to TensorBoard, and save videos whenever a new best episode reward is reached.

In [1]:
# Install/upgrade dependencies if needed
# Uncomment if your environment does not already have these packages installed.
# !pip install -U stable-baselines3==2.3.2 gymnasium==0.29.1 numpy==1.26.4 protobuf==3.20.3 tensorboard==2.14.0

In [2]:
import os
import sys
from types import SimpleNamespace

import gymnasium as gym
import numpy as np

# Make the local project importable so the custom environment can be imported directly.
sys.path.insert(0, r'C:/Users/16469/Desktop/circuit/racetrack-agents')

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecMonitor

from racetrack_env import RaceTrackEnv


c:\Users\16469\anaconda3\envs\circuit\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


## 1. Environment wrapper

The original project used a custom `RaceTrackEnv` that was written for an older HighwayEnv API. SB3 expects a standard Gym/Gymnasium interface, so we wrap the environment with a small adapter that exposes `reset()`/`step()` in the modern form.

In [4]:
class SB3RaceTrackEnv(RaceTrackEnv):
    """Compatibility wrapper for SB3 training on the original race-track environment."""

    def _is_terminated(self):
        # The original environment ends an episode when the car crashes, reaches the goal,
        # exceeds the maximum number of steps, or leaves the road.
        return self.vehicle.crashed or self._is_goal() or self.steps >= self.config['duration'] or not self.vehicle.on_road

    def _is_truncated(self):
        # The original environment does not use a separate truncation signal, so we keep it False.
        return False

    def reset(self, *, seed=None, options=None):
        reset_result = super().reset(seed=seed, options=options)
        if isinstance(reset_result, tuple):
            obs, info = reset_result
        else:
            obs, info = reset_result, {}
        self.observation = obs
        return obs, info

    def step(self, action):
        # Forward actions to the original environment and preserve the Gymnasium-style tuple.
        obs, reward, terminated, truncated, info = super().step(action)
        return obs, reward, terminated, truncated, info


class DiscreteSteeringWrapper(gym.Wrapper):
    """Maps a discrete action set to continuous steering commands for DQN."""

    def __init__(self, env):
        super().__init__(env)
        self.action_space = gym.spaces.Discrete(3)

    def step(self, action):
        if isinstance(action, np.ndarray):
            action = int(action.item())
        # DQN uses three actions: left, straight, right.
        steering = {0: -0.5, 1: 0.0, 2: 0.5}[int(action)]
        # The environment's continuous action is [steering, throttle], so throttle is held at zero.
        obs, reward, terminated, truncated, info = self.env.step([steering, 0.0])
        return obs, reward, terminated, truncated, info


def make_env(spawn_vehicles=0, random_lane=False, all_random=False, num_actions=2, discrete=False):
    def _init():
        opt = SimpleNamespace(
            obs_dim=(2, 18, 18),
            num_actions=num_actions,
            all_random=all_random,
            spawn_vehicles=spawn_vehicles,
            random_lane=random_lane,
            offroad_thres=0,
            random_obstacles=0,
        )
        env = SB3RaceTrackEnv(opt)
        if discrete:
            env = DiscreteSteeringWrapper(env)
        return env

    return _init


## 2. Build training environment

SB3 trains more stably with vectorized environments. We create a single wrapped environment for demonstration purposes.

In [3]:
exp_id = 'dqn2'
log_dir = os.path.join(r'C:/Users/16469/Desktop/circuit/racetrack-agents/logs', exp_id)
tensorboard_dir = os.path.join(log_dir, 'tensorboard')
os.makedirs(log_dir, exist_ok=True)
os.makedirs(tensorboard_dir, exist_ok=True)

# PPO uses a continuous action vector of the form [steering, throttle].
# We build a single vectorized environment so SB3 can collect rollouts efficiently.
env = DummyVecEnv([make_env(spawn_vehicles=0, random_lane=False, all_random=False, num_actions=2, discrete=False)])
env = VecNormalize(env, norm_obs=True, norm_reward=False, clip_obs=10.0)

print('Observation space:', env.observation_space)
print('Action space:', env.action_space)


NameError: name 'make_env' is not defined

## 3. PPO training

PPO is a good default for continuous control. The original project used a custom PPO agent, but SB3 provides a robust implementation with clipping and GAE built in.

In [10]:
class EpisodeRewardCallback(BaseCallback):
    """Log episode reward and save videos when a new best reward is reached."""

    def __init__(self, log_dir: str, verbose=0):
        super().__init__(verbose)
        self.log_dir = log_dir
        self.best_reward = -np.inf
        self.video_dir = os.path.join(log_dir, 'videos')
        os.makedirs(self.video_dir, exist_ok=True)

    def _on_step(self) -> bool:
        infos = self.locals.get('infos', [])
        if infos:
            for info in infos:
                if 'episode' in info:
                    ep_reward = float(info['episode']['r'])
                    self.logger.record('rollout/ep_reward', ep_reward)
                    if ep_reward > self.best_reward:
                        self.best_reward = ep_reward
                        self.logger.record('rollout/new_best_episode', ep_reward)
                        print(f'New best episode reward: {ep_reward:.2f}')
        return True


ppo_model = PPO(
    policy='MlpPolicy',
    env=env,
    policy_kwargs=dict(net_arch=[256, 256]),
    learning_rate=2.5e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    verbose=1,
    tensorboard_log=tensorboard_dir,
    device='auto',
)

ppo_model.learn(
    total_timesteps=60000,
    callback=[EpisodeRewardCallback(log_dir)],
    progress_bar=False,
)


Using cuda device
Logging to C:/Users/16469/Desktop/circuit/racetrack-agents/logs/sb3_demo\tensorboard\PPO_3
-----------------------------
| time/              |      |
|    fps             | 49   |
|    iterations      | 1    |
|    time_elapsed    | 41   |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 49          |
|    iterations           | 2           |
|    time_elapsed         | 82          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.036208138 |
|    clip_fraction        | 0.292       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.83       |
|    explained_variance   | -0.0215     |
|    learning_rate        | 0.00025     |
|    loss                 | 2.98        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0564     |
|    std 

## 4. DQN training

DQN is suitable for discrete actions. The original repository had a discrete-action version of the task, so this section mirrors that idea with a simpler action space.

In [6]:
# DQN uses the same discrete steering setup as the original repository.
# The hyperparameters below follow the provided command as closely as possible in SB3.
dqn_env = DummyVecEnv([make_env(spawn_vehicles=3, random_lane=False, all_random=False, num_actions=1, discrete=True)])
dqn_env = VecMonitor(dqn_env, os.path.join(log_dir, 'dqn'))
dqn_env = VecNormalize(dqn_env, norm_obs=True, norm_reward=False, clip_obs=10.0)

dqn_model = DQN(
    policy='MlpPolicy',
    env=dqn_env,
    policy_kwargs=dict(net_arch=[64, 64, 64]),
    learning_rate=5e-5,
    buffer_size=10000,
    learning_starts=500,
    batch_size=256,
    gamma=0.99,
    exploration_initial_eps=0.6,
    exploration_final_eps=0.0,
    exploration_fraction=0.5,
    target_update_interval=1000,
    train_freq=1,
    gradient_steps=1,
    verbose=1,
    tensorboard_log=tensorboard_dir,
    device='auto',
)


dqn_model.learn(total_timesteps=250000, progress_bar=False)

Using cuda device
Logging to C:/Users/16469/Desktop/circuit/racetrack-agents/logs/sb3_demo\tensorboard\DQN_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 67       |
|    ep_rew_mean      | 10.9     |
|    exploration_rate | 0.936    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 58       |
|    time_elapsed     | 4        |
|    total_timesteps  | 268      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 67       |
|    ep_rew_mean      | 12.3     |
|    exploration_rate | 0.873    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 59       |
|    time_elapsed     | 9        |
|    total_timesteps  | 536      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 67       |
|    ep_rew_mean

## 5. Save models

In [16]:
dqn_path = os.path.join(log_dir, 'dqn_racetrack')
dqn_model.save(dqn_path)

print('Saved DQN model to', dqn_path)

Saved PPO model to C:/Users/16469/Desktop/circuit/racetrack-agents/logs/sb3_demo\ppo_racetrack
Saved DQN model to C:/Users/16469/Desktop/circuit/racetrack-agents/logs/sb3_demo\dqn_racetrack


## 6. Quick rollout

In [ ]:
%pip install imageio[ffmpeg] onnxscript

import imageio
import os


def run_rollout(model, env_factory, steps=200, video_path=None):
    """Run a rollout and optionally save the episode frames as a video."""
    env = env_factory()
    env.render_mode = 'rgb_array'
    obs, info = env.reset()
    total_reward = 0.0
    frames = []
    for _ in range(steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if video_path is not None:
            frame = env.render()
            if frame is not None:
                frames.append(frame)
        if terminated or truncated:
            break
    if video_path is not None and frames:
        imageio.mimsave(video_path, frames, fps=10)
    return total_reward


def evaluate_and_record_best_videos(model, env_factory, episodes=3, steps=200, video_dir=None):
    """Run several evaluation episodes and save videos only when a new best reward is reached."""
    video_dir = video_dir or os.path.join(log_dir, 'videos')
    os.makedirs(video_dir, exist_ok=True)
    best_reward = -float('inf')
    for episode_idx in range(episodes):
        reward = run_rollout(model, env_factory, steps=steps, video_path=None)
        print(f'Evaluation episode {episode_idx + 1}: reward={reward:.2f}')
        if reward > best_reward:
            best_reward = reward
            video_path = os.path.join(video_dir, f'best_episode_{episode_idx + 1:02d}_{reward:.1f}.mp4')
            run_rollout(model, env_factory, steps=steps, video_path=video_path)
            print(f'Saved best-episode video to {video_path}')
    return best_reward


rollout_env = lambda: make_env(spawn_vehicles=3, random_lane=False, all_random=False, num_actions=1, discrete=True)()
rollout_reward = evaluate_and_record_best_videos(
    dqn_model,
    rollout_env,
    episodes=3,
    steps=200,
    video_dir=os.path.join(log_dir, 'videos'),
)
print('Best DQN evaluation reward:', rollout_reward)


Note: you may need to restart the kernel to use updated packages.


    opencv-python (>=3.) ; extra == 'all'
                  ~~~~^


Evaluation episode 1: reward=38.23


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1000, 1000) to (1008, 1008) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Saved best-episode video to C:/Users/16469/Desktop/circuit/racetrack-agents/logs/sb3_demo\videos\best_episode_01_38.2.mp4
Evaluation episode 2: reward=38.23
Evaluation episode 3: reward=38.23
Best PPO evaluation reward: 38.22994806907457


C:\Users\16469\AppData\Local\Temp\ipykernel_15508\1540343859.py:80: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `ActorWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ActorWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX model to C:/Users/16469/Desktop/circuit/racetrack-agents/logs/sb3_demo\ppo_racetrack.onnx


c:\Users\16469\anaconda3\envs\circuit\lib\copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


: 

In [ ]:
env.close()

: 

# Summary of changes

- The original project trains custom TensorFlow agents directly in `main.py`.
- This notebook replaces that with SB3's PPO and DQN implementations.
- The original custom environment is wrapped so SB3 can use it without rewriting the race-track logic.
- PPO is used for continuous steering-style control, while DQN is used for the discrete-action variant.
- The notebook also saves models and provides a simple rollout evaluation.